# 00 · Project overview

**Goal of the thesis.** Detect malicious activity at two complementary layers of a monitored host and
measure how robust the detectors are when the data drifts over time or an attacker actively tries to evade them.

| Layer | Data | What one sample is | Detector |
|---|---|---|---|
| **Network (NIDS)** | CIC-IDS-2017 flow records | a *window* of 500 consecutive flows → one graph (flows = nodes, shared IP / port / temporal adjacency = edges) | MFGraph GNN |
| **Host (PE)** | EMBER 2018 static PE features | one executable → one graph (9 feature groups = nodes, fixed semantic edges) | MFGraph GNN |

Both detectors use the same architecture, **MFGraph** (a DGCNN: 3 GCN layers → SortPooling → 1D conv → MLP).
Each domain is compared with **classical ML baselines** (RF, DT, SVM, KNN, LR, AdaBoost, MLP, FFDNN) trained on the
*flattened* version of the same graphs (mean-pooled node features), so the comparison isolates the value of the graph structure.

A **Dropout-WGAN-GP** produces synthetic malicious graphs. They are used in two ways:
1. to **augment training** (adversarially hardened classifier);
2. to **generate evasive perturbations** for the PE robustness test.

## Research questions
1. **Performance** – Does the graph representation match or beat classical baselines on clean data?
2. **Concept drift** – How does performance degrade on data from *later* than the training period?
3. **Adversarial robustness** – How much does an attacker who perturbs traffic timing / packet sizes, or GAN-shifts PE features, degrade each model?
4. **Defense in depth** – When both detectors are combined in one alerting pipeline, how many threats are stopped and at which layer?

## Threat model (summary)
* The attacker controls **only their own malicious samples** (their flows / their executable), not the training data (no poisoning).
* **Network:** can stretch or compress packet sizes and inter-arrival times (padding, throttling) while the attack still works.
* **Host:** can modify the binary so that its static features move toward the benign distribution (simulated with GAN noise of growing intensity).
* The attacker is **black-box**: no gradients of the detector are used.

## Notebook map
| # | Notebook | Answers |
|---|---|---|
| 01 | `01_data_exploration_EMBER`, `01_data_exploration_CIC` | What the data looks like, class balance, data-quality issues |
| 02 | `02_graph_construction` | How raw samples become graphs (both domains) |
| 03 | `03_training_analysis` | Do GNN and GAN training converge? Over/under-fitting? |
| 04 | `04_clean_performance_and_drift` | RQ1 + RQ2 |
| 05 | `05_adversarial_robustness` | RQ3 |
| 06 | `06_unified_pipeline` | RQ4 |

## Reproducing the results
Every notebook only *reads* artefacts in `experiments/`. Produce them with the orchestrator:

```bash
python orchestrator.py -t build_cicids          # NIDS graphs
python orchestrator.py -t ember_graph           # PE graphs (train = Jan 2018, test = Feb–Dec 2018)
python orchestrator.py -t train_pe_gnn          # and: train_nids_gnn, train_pe_gan, train_nids_gan
python orchestrator.py -t train_pe_baselines    # and: train_nids_baselines
python orchestrator.py -t eval_drift
python orchestrator.py -t eval_pe_adversarial   # and: eval_nids_adversarial
python orchestrator.py -t unified_pipeline
```